In [1]:
######################################################################################
# Authored by Andrew Wan Ju Kang, Ph.D.
# soarhigh0714@gmail.com
# Do not re-distribute.
######################################################################################

## 1. Tool 정의 (ReAct 에이전트용)

LangChain v1.0에서 `@tool` 데코레이터를 사용하여 도구를 정의합니다.

- 함수의 **docstring**이 도구 설명으로 사용됩니다.
- `args_schema`로 Pydantic 모델을 지정하면 입력 유효성 검증이 가능합니다.

참조: [LangChain v1.0 Tools](https://docs.langchain.com/oss/python/langchain/tools)

In [2]:
from langchain.tools import tool
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    """날씨 조회를 위한 입력 스키마"""
    location: str = Field(description="날씨를 조회할 도시 이름 (예: '서울', 'Tokyo')")


@tool("get_weather", args_schema=WeatherInput)
def get_weather(location: str) -> str:
    """주어진 도시의 현재 날씨 정보를 조회합니다.
    도시 이름을 입력하면 온도, 상태, 습도를 반환합니다."""
    weather_data = {
        "서울": {"temp": "22°C", "condition": "맑음", "humidity": "45%"},
        "부산": {"temp": "25°C", "condition": "흐림", "humidity": "70%"},
        "도쿄": {"temp": "28°C", "condition": "비", "humidity": "80%"},
        "Tokyo": {"temp": "28°C", "condition": "Rainy", "humidity": "80%"},
        "뉴욕": {"temp": "18°C", "condition": "구름 조금", "humidity": "55%"},
        "New York": {"temp": "18°C", "condition": "Partly Cloudy", "humidity": "55%"},
    }
    if location in weather_data:
        d = weather_data[location]
        return f"{location} 날씨: 온도 {d['temp']}, 상태 {d['condition']}, 습도 {d['humidity']}"
    return f"{location}의 날씨 정보를 찾을 수 없습니다. 사용 가능 도시: {', '.join(weather_data.keys())}"


@tool("get_user_location")
def get_user_location() -> str:
    """현재 사용자의 위치를 조회합니다.
    별도 입력 없이 호출하면 사용자의 현재 도시를 반환합니다."""
    return "서울"


class CalculatorInput(BaseModel):
    """계산기 입력 스키마"""
    expression: str = Field(description="계산할 수학 표현식 (예: '2 + 3 * 4')")


@tool("calculator", args_schema=CalculatorInput)
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. 사칙연산, 제곱, 괄호 등을 지원합니다."""
    try:
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "오류: 허용되지 않는 문자가 포함되어 있습니다."
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


# 사용할 도구 리스트
tools = [get_weather, get_user_location, calculator]

print(f"🔧 등록된 도구 ({len(tools)}개): {[t.name for t in tools]}")

🔧 등록된 도구 (3개): ['get_weather', 'get_user_location', 'calculator']


/opt/conda/lib/python3.10/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. LangChain v1.0 `create_agent`로 에이전트 구성

### ✅ 핵심 포인트

| 항목 | 설명 |
|---|---|
| **import** | `from langchain.agents import create_agent` |
| **에이전트 생성** | `create_agent(model=llm, tools=tools, system_prompt=...)` |
| **필요 패키지** | `langchain>=1.0`, `langchain-openai` |
| **LangGraph** | 내부적으로 사용되지만 직접 import 불필요 |

> `create_agent`는 내부적으로 LangGraph 런타임 위에서 동작하지만,
> 사용자가 LangGraph API를 직접 다룰 필요는 없습니다.
>
> 로컬 llama-server는 OpenAI 호환 API를 제공하므로,
> `ChatOpenAI`에 `base_url`을 지정하여 연결합니다.
>
> 참조:
> - [LangChain v1.0 Agents](https://docs.langchain.com/oss/python/langchain/agents)
> - [create_agent API](https://reference.langchain.com/python/langchain/agents/factory/create_agent)
> - [Chat Model Integrations](https://docs.langchain.com/oss/python/integrations/chat)

In [3]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# 서버가 실행될 호스트 주소를 설정합니다.
SERVER_HOST = "127.0.0.1"
# 서버가 사용할 포트 번호를 설정합니다.
SERVER_PORT = 8181

# ─── LLM 연결: llama-server → OpenAI 호환 API ───
llm = ChatOpenAI(
    model="gemma4",                                       # 모델명 (llama-server에서는 무시됨)
    base_url=f"http://{SERVER_HOST}:{SERVER_PORT}/v1",          # 로컬 llama-server
    api_key="not-needed",                                       # 로컬 서버이므로 불필요
    temperature=0.1,
    max_tokens=1024,
)

# ─── 시스템 프롬프트 ───
SYSTEM_PROMPT = """당신은 도구(Tools)를 활용하여 사용자에게 도움을 주는 유능한 AI 어시스턴트입니다. 
다음의 지침을 엄격히 준수하여 응답하십시오.

1. 도구 사용 지침

날씨 관련 문의: 사용자가 날씨를 물어보면 제공된 도구를 사용하십시오.
사용자의 위치 정보를 먼저 파악해야 할 경우, get_user_location을 호출하십시오.
확보된 위치 정보를 바탕으로 get_weather를 호출하여 정보를 제공하십시오.
수학 관련 문의: 수학적인 계산이나 문제 풀이가 필요한 경우, 반드시 calculator 도구를 사용하십시오.

2. 응답 형식 및 언어

모든 응답은 한국어로 작성하십시오.
논리적이고 정확한 답변을 위해 단계별로 차근차근(Think step by step) 사고하여 결론을 도출하십시오."""

# ─── ReAct 에이전트 생성 ───
# create_agent 시그니처 (공식 API Reference):
#   create_agent(
#       model: str | BaseChatModel,
#       tools: Sequence[BaseTool | Callable | dict] | None,
#       *, system_prompt, middleware, response_format, ...
#   ) -> CompiledStateGraph
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ LangChain v1.0 ReAct 에이전트 생성 완료!")

✅ LangChain v1.0 ReAct 에이전트 생성 완료!


## 3. 에이전트 실행 함수

두 가지 실행 방식을 제공합니다:

| 방식 | 함수 | 설명 |
|---|---|---|
| **invoke** | `run_agent()` | 전체 결과를 한 번에 받음 |
| **stream** | `run_agent_stream()` | 단계별 스트리밍 |

참조: [Agents - Invocation](https://docs.langchain.com/oss/python/langchain/agents)

In [4]:
import base64
import mimetypes


def run_agent(query: str, image_url: str = None, verbose: bool = True):
    """에이전트를 invoke 방식으로 실행합니다.

    Args:
        query: 질문 텍스트
        image_url: 이미지 URL 또는 로컬 파일 경로 (선택, 멀티모달)
        verbose: 중간 단계 출력 여부
    """
    print(f"\n{'='*60}")
    print(f"📝 질문: {query}")
    if image_url:
        print(f"🖼️ 이미지: {image_url}")
    print(f"{'='*60}")

    # ─── 메시지 구성 ───
    if image_url:
        if not image_url.startswith("http"):
            mime = mimetypes.guess_type(image_url)[0] or "image/png"
            with open(image_url, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            url = f"data:{mime};base64,{b64}"
        else:
            url = image_url

        user_message = {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": query},
            ],
        }
    else:
        user_message = {"role": "user", "content": query}

    # ─── 에이전트 실행 (invoke) ───
    result = agent.invoke({"messages": [user_message]})

    # ─── 결과 출력 ───
    if verbose:
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🔧 Tool Call: {tc['name']}({tc['args']})")
            elif hasattr(msg, "type") and msg.type == "tool":
                print(f"  📦 Tool Result [{msg.name}]: {msg.content}")

    final_response = result["messages"][-1].content
    print(f"\n🎯 최종 응답:\n{final_response}")
    return final_response


def run_agent_stream(query: str, image_url: str = None, verbose: bool = True):
    """에이전트를 stream 방식으로 실행합니다.

    Args:
        query: 질문 텍스트
        image_url: 이미지 URL 또는 로컬 파일 경로 (선택, 멀티모달)
        verbose: 중간 단계 출력 여부
    """
    print(f"\n{'='*60}")
    print(f"📝 질문 (스트리밍): {query}")
    if image_url:
        print(f"🖼️ 이미지: {image_url}")
    print(f"{'='*60}")

    # ─── 메시지 구성 ───
    if image_url:
        if not image_url.startswith("http"):
            mime = mimetypes.guess_type(image_url)[0] or "image/png"
            with open(image_url, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            url = f"data:{mime};base64,{b64}"
        else:
            url = image_url

        user_message = {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": url}},
                {"type": "text", "text": query},
            ],
        }
    else:
        user_message = {"role": "user", "content": query}

    # ─── 에이전트 스트리밍 실행 ───
    step = None
    for step in agent.stream(
        {"messages": [user_message]},
        stream_mode="values",
    ):
        last_msg = step["messages"][-1]

        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                if verbose:
                    print(f"  🔧 Tool Call: {tc['name']}({tc['args']})")
        elif hasattr(last_msg, "type") and last_msg.type == "tool":
            if verbose:
                print(f"  📦 Tool Result [{last_msg.name}]: {last_msg.content}")

    if step:
        final_response = step["messages"][-1].content
        print(f"\n🎯 최종 응답:\n{final_response}")
        return final_response
    return None


print("✅ 실행 함수 정의 완료!")

✅ 실행 함수 정의 완료!


## 4. 실행 예시

### 4-1. 텍스트 질문 (invoke 방식)

In [5]:
run_agent("서울 날씨 어때?")


📝 질문: 서울 날씨 어때?
  🔧 Tool Call: get_weather({'location': '서울'})
  📦 Tool Result [get_weather]: 서울 날씨: 온도 22°C, 상태 맑음, 습도 45%

🎯 최종 응답:
서울 날씨는 현재 온도 22°C, 상태는 맑음, 습도는 45%입니다.


'서울 날씨는 현재 온도 22°C, 상태는 맑음, 습도는 45%입니다.'

### 4-2. 텍스트 질문 (streaming 방식)

In [6]:
run_agent_stream("현재 내 위치의 날씨를 알려줘")


📝 질문 (스트리밍): 현재 내 위치의 날씨를 알려줘
  🔧 Tool Call: get_user_location({})
  📦 Tool Result [get_user_location]: 서울
  🔧 Tool Call: get_weather({'location': '서울'})
  📦 Tool Result [get_weather]: 서울 날씨: 온도 22°C, 상태 맑음, 습도 45%

🎯 최종 응답:
현재 사용자님의 위치는 **서울**이며, 현재 서울의 날씨는 **온도 22°C, 상태 맑음, 습도 45%** 입니다.


'현재 사용자님의 위치는 **서울**이며, 현재 서울의 날씨는 **온도 22°C, 상태 맑음, 습도 45%** 입니다.'

### 4-3. 계산기 도구 사용

In [7]:
run_agent("(15 + 27) * 3 은 얼마야?")


📝 질문: (15 + 27) * 3 은 얼마야?
  🔧 Tool Call: calculator({'expression': '(15 + 27) * 3'})
  📦 Tool Result [calculator]: (15 + 27) * 3 = 126

🎯 최종 응답:
Think step by step:
1.  **사용자 요청 분석:** 사용자는 수학 문제 "(15 + 27) * 3"의 답을 묻고 있습니다.
2.  **도구 선택:** 이 요청은 수학적인 계산을 요구하므로, `calculator` 도구를 사용해야 합니다.
3.  **도구 호출:** `calculator` 도구에 계산할 표현식 `(15 + 27) * 3`을 전달하여 계산을 요청합니다.
4.  **도구 응답 처리:** `calculator` 도구는 계산 결과를 `126`으로 반환했습니다.
5.  **최종 답변 생성:** 계산 결과를 사용자에게 명확하게 전달합니다.

답변: (15 + 27) * 3은 126입니다.


'Think step by step:\n1.  **사용자 요청 분석:** 사용자는 수학 문제 "(15 + 27) * 3"의 답을 묻고 있습니다.\n2.  **도구 선택:** 이 요청은 수학적인 계산을 요구하므로, `calculator` 도구를 사용해야 합니다.\n3.  **도구 호출:** `calculator` 도구에 계산할 표현식 `(15 + 27) * 3`을 전달하여 계산을 요청합니다.\n4.  **도구 응답 처리:** `calculator` 도구는 계산 결과를 `126`으로 반환했습니다.\n5.  **최종 답변 생성:** 계산 결과를 사용자에게 명확하게 전달합니다.\n\n답변: (15 + 27) * 3은 126입니다.'

### 4-4. 이미지 + 텍스트 (멀티모달)

In [8]:
run_agent(
    "이 이미지에 무엇이 보이는지 설명해줘",
    image_url="/home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg",
)


📝 질문: 이 이미지에 무엇이 보이는지 설명해줘
🖼️ 이미지: /home/user/MultiAgents/Day_2/Ex/image_data/cat.jpg

🎯 최종 응답:
이 이미지에는 편안하게 쉬고 있는 **고양이**가 보입니다.

**주요 특징:**

*   **고양이:** 얼룩무늬(탭) 무늬를 가진 고양이가 중앙에 누워 있습니다. 털은 검은색, 회색, 갈색 줄무늬가 섞여 있으며, 발 부분은 하얗습니다. 고양이는 매우 편안한 자세로 몸을 늘어뜨리고 있습니다.
*   **배경/소품:** 고양이는 부드럽고 질감이 느껴지는 **밝은 색의 담요나 수건** 위에 누워 있습니다. 이 담요는 나무 재질의 표면(바닥이나 테이블) 위에 놓여 있습니다.
*   **환경:** 왼쪽에는 **큰 창문**이 보이며, 창밖으로는 푸른 나뭇잎과 야외 풍경이 비쳐 들어와 자연광이 실내로 들어오고 있습니다.

전반적으로 이 이미지는 따뜻하고 아늑하며, 고양이가 햇살 좋은 창가에서 휴식을 취하는 평화로운 순간을 포착하고 있습니다.


'이 이미지에는 편안하게 쉬고 있는 **고양이**가 보입니다.\n\n**주요 특징:**\n\n*   **고양이:** 얼룩무늬(탭) 무늬를 가진 고양이가 중앙에 누워 있습니다. 털은 검은색, 회색, 갈색 줄무늬가 섞여 있으며, 발 부분은 하얗습니다. 고양이는 매우 편안한 자세로 몸을 늘어뜨리고 있습니다.\n*   **배경/소품:** 고양이는 부드럽고 질감이 느껴지는 **밝은 색의 담요나 수건** 위에 누워 있습니다. 이 담요는 나무 재질의 표면(바닥이나 테이블) 위에 놓여 있습니다.\n*   **환경:** 왼쪽에는 **큰 창문**이 보이며, 창밖으로는 푸른 나뭇잎과 야외 풍경이 비쳐 들어와 자연광이 실내로 들어오고 있습니다.\n\n전반적으로 이 이미지는 따뜻하고 아늑하며, 고양이가 햇살 좋은 창가에서 휴식을 취하는 평화로운 순간을 포착하고 있습니다.'